# Scene Analysis with Computer Vision and Machine Learning

This notebook builds an inspectable pipeline that converts an image into object detections,
a scene graph, a natural-language description, answers to structured questions, and a
zone-monitoring decision.

**Workflow:** synthetic data → pixel colour model → segmentation → shape/size classifier →
scene graph → language outputs → evaluation.

The experiments in this notebook are newly executed by the user. They should not be
assumed to reproduce any previously reported numerical results.

## 1. Environment and imports

Install the dependencies once if needed:
`pip install numpy opencv-python scikit-learn matplotlib pandas`

In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from scene_analysis import (
    SceneAnalyzer, generate_scene, build_training_set,
    draw_detections, evaluate_noise_sweep, COLORS
)

print("Python:", sys.version.split()[0])
print("OpenCV:", cv2.__version__)

## 2. Generate and inspect a synthetic scene

The generator creates 320 × 240 scenes with 2–6 coloured geometric objects.
It also provides an integer label map, which acts as exact ground truth for training
and evaluation. Noise and illumination can be controlled independently.

In [ ]:
sample = generate_scene(seed=7, noise_sigma=10, illumination=1.0)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(sample.image)
axes[0].set_title("Generated RGB scene")
axes[1].imshow(sample.label_map, cmap="tab20")
axes[1].set_title("Ground-truth object labels")
for ax in axes: ax.axis("off")
plt.show()

pd.DataFrame([vars(o) for o in sample.objects])

## 3. Create the training set

For a quick run, start with 200–300 scenes. Increase the number for a more stable
model, especially when running a full noise sweep.

In [ ]:
training_scenes = build_training_set(n_scenes=250, seed=100)
print("Training scenes:", len(training_scenes))
print("Mean visible objects:", np.mean([len(s.objects) for s in training_scenes]))

## 4. Train the perception models

The pixel model uses CIE Lab colour features and Quadratic Discriminant Analysis.
Connected components produce object candidates. A Random Forest classifies each
candidate into one of six joint shape/size labels. Object confidence combines the
mean colour posterior and the form-class probability.

In [ ]:
analyzer = SceneAnalyzer(random_state=42)
start = time.perf_counter()
analyzer.fit(training_scenes)
print(f"Training completed in {time.perf_counter()-start:.2f} seconds")

## 5. Run the complete pipeline

The same detected objects feed the caption generator, question-answering component,
and zone-monitoring rule. This shared representation makes the outputs traceable.

In [ ]:
test_scene = generate_scene(seed=2026, noise_sigma=20)
question = "How many small orange objects are there?"
zone = (100, 70, 230, 190)

result = analyzer.analyze(
    test_scene.image,
    question=question,
    target_color="orange",
    zone=zone,
    threshold=0.0
)

print("CAPTION:", result["caption"])
print("QUESTION:", question)
print("ANSWER:", result["answer"])
print("DECISION:", result["decision"]["explanation"])

visual = draw_detections(test_scene.image, result["detections"], zone=zone)
plt.figure(figsize=(9, 6))
plt.imshow(visual)
plt.title("Detected objects and restricted zone")
plt.axis("off")
plt.show()

pd.DataFrame([vars(o) for o in result["detections"]])

## 6. Inspect the scene graph

Nodes store object attributes, bounding boxes, centres, and confidence.
Edges encode spatial relations using a 20-pixel margin.

In [ ]:
import json
print(json.dumps(result["scene_graph"], indent=2))

## 7. Try different questions

The parser supports existence, counting, attribute, spatial, comparison, and
extreme-object question patterns. It intentionally returns `unknown` when the
question does not fit the supported patterns or is ambiguous.

In [ ]:
questions = [
    "Is there a red triangle?",
    "How many small orange objects are there?",
    "What shape is the large purple object?",
    "Are there more red than blue objects?",
    "What colour is the leftmost object?"
]
for q in questions:
    print(f"{q} -> {analyzer.answer_question(result['detections'], q)}")

## 8. Confidence-aware descriptions

Increasing the confidence threshold suppresses uncertain detections. This can
reduce false mentions, but it can also omit real objects. Inspect the trade-off.

In [ ]:
thresholds = [0.0, 0.3, 0.5, 0.7, 0.85]
for threshold in thresholds:
    caption = analyzer.caption(result["detections"], confidence_threshold=threshold)
    kept = sum(o.confidence >= threshold for o in result["detections"])
    print(f"threshold={threshold:.2f} | retained={kept} | {caption}")

## 9. Evaluate robustness to sensor noise

This compact sweep uses fresh generated scenes. It reports object-level precision,
recall, and F1 using IoU-based matching. The default number of test scenes is kept
small so that it runs on an ordinary CPU; increase it for a more reliable estimate.

In [ ]:
noise_results = evaluate_noise_sweep(
    analyzer,
    noise_levels=(0, 20, 40, 60, 80, 100),
    n_scenes=30
)
noise_df = pd.DataFrame(noise_results)
display(noise_df)

plt.figure(figsize=(8, 5))
for metric in ("precision", "recall", "f1"):
    plt.plot(noise_df["noise_sigma"], noise_df[metric], marker="o", label=metric)
plt.xlabel("Gaussian noise standard deviation (σ)")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.title("Detection performance under sensor noise")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 10. Test on your own image

Set `IMAGE_PATH` to an image file available in the notebook environment. The model
was trained on simple synthetic shapes, so photographs may produce poor detections.
This is an expected domain limitation, not evidence of real-world object recognition.

In [ ]:
IMAGE_PATH = ""  # Example: "my_image.png"

if IMAGE_PATH:
    bgr = cv2.imread(IMAGE_PATH)
    if bgr is None:
        raise FileNotFoundError(f"Could not read: {IMAGE_PATH}")
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    own_result = analyzer.analyze(rgb)
    print(own_result["caption"])
    plt.figure(figsize=(10, 6))
    plt.imshow(draw_detections(rgb, own_result["detections"]))
    plt.axis("off")
    plt.show()
else:
    print("Add an image path to IMAGE_PATH to test a local image.")

## 11. Export predictions

The detected scene graph can be saved as JSON for later inspection or integration
with another application.

In [ ]:
with open("scene_graph.json", "w", encoding="utf-8") as f:
    json.dump(result["scene_graph"], f, indent=2)
print("Saved scene_graph.json")

## 12. Interpretation and limitations

- The implementation uses a controlled synthetic dataset, not a public photograph dataset.
- The question parser uses keywords and fixed patterns; it is not a general language model.
- Same-colour objects that touch can merge during connected-component segmentation.
- The synthetic training domain is simple; photographs require a stronger detector and
  training data representative of the target environment.
- Results from this notebook must be measured by running its cells. Do not copy metrics
  from another experiment and describe them as results from this run.